# Member 2 — Exploratory Data Analysis
## Amazon ML Challenge 2026 · Business Entity Resolution · Team Akatsuki

**Responsibility:** Understand the distribution and characteristics of the business records across all three sources.

---

**Key question:** *What kinds of noise and variation actually exist in this dataset?*

This notebook produces four plots and one HTML report saved into `member2/`.

In [ ]:
import json
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick

warnings.filterwarnings('ignore')

# ── Paths ──────────────────────────────────────────────────────────────────
BASE   = Path('../../..')              # amazon ml challenge/
DATA   = BASE / 'student_resource' / 'dataset'
TRAIN  = DATA / 'train'
TEST   = DATA / 'test'
OUT    = Path('.')                     # member2/
PLOTS  = OUT / 'plots'
PLOTS.mkdir(parents=True, exist_ok=True)

# ── Style ──────────────────────────────────────────────────────────────────
PALETTE = ['#6C63FF', '#FF6584', '#43B89C', '#FFBE0B', '#FB5607']
plt.rcParams.update({
    'figure.facecolor': '#0F0F1A', 'axes.facecolor': '#1A1A2E',
    'axes.edgecolor': '#3A3A5C',   'text.color': '#E0E0FF',
    'axes.labelcolor': '#E0E0FF',  'xtick.color': '#A0A0CC',
    'ytick.color': '#A0A0CC',      'grid.color': '#2A2A4A',
    'grid.linestyle': '--',        'grid.alpha': 0.5,
    'font.family': 'sans-serif',   'axes.titlesize': 13,
    'axes.labelsize': 11,
})
print('Libraries loaded.')

---
## 1. Load All Data Sources

In [ ]:
print('Loading train data ...')
s1_tr = pd.read_csv(TRAIN / 'train_source1.tsv', sep='\t')
s2_tr = pd.read_csv(TRAIN / 'train_source2.tsv', sep='\t')
s3_tr = pd.read_csv(TRAIN / 'train_source3.tsv', sep='\t')
gt_tr = pd.read_csv(TRAIN / 'train_ground_truth.tsv', sep='\t')

print('Loading test data ...')
s1_te = pd.read_csv(TEST / 'test_source1.tsv',  sep='\t')
s2_te = pd.read_csv(TEST / 'test_source2.tsv',  sep='\t')
s3_te = pd.read_csv(TEST / 'test_source3.tsv',  sep='\t')
print('All data loaded.')

---
## 2. Row Counts & Basic Shape

In [ ]:
rc = {
    'Train S1': len(s1_tr), 'Train S2': len(s2_tr), 'Train S3': len(s3_tr),
    'Test S1':  len(s1_te), 'Test S2':  len(s2_te), 'Test S3':  len(s3_te),
}
for k, v in rc.items():
    print(f'  {k}: {v:,} records')

print(f"\nTrain total: {rc['Train S1']+rc['Train S2']+rc['Train S3']:,}")
print(f"Test  total: {rc['Test S1'] +rc['Test S2'] +rc['Test S3']:,}")

---
## 3. Missing Value Analysis

In [ ]:
sources = [
    ('S1_train', s1_tr), ('S2_train', s2_tr), ('S3_train', s3_tr),
    ('S1_test',  s1_te), ('S2_test',  s2_te), ('S3_test',  s3_te),
]
fields = ['business_name', 'business_address', 'country']

mv = {}
for label, df in sources:
    mv[label] = {f: round(df[f].isna().mean()*100, 2) for f in fields}

mv_df = pd.DataFrame(mv, index=fields).T
print('Missing value percentages:')
print(mv_df.to_string())

# Plot: Missing Values Heatmap
import numpy as np
mv_matrix = np.array([[mv[src][f] for f in fields] for src, _ in sources])
src_labels = [s for s, _ in sources]

fig, ax = plt.subplots(figsize=(9, 5))
im = ax.imshow(mv_matrix, cmap='RdPu', aspect='auto', vmin=0, vmax=max(mv_matrix.max(), 1))
ax.set_xticks(range(len(fields)))
ax.set_xticklabels(fields, rotation=15, ha='right')
ax.set_yticks(range(len(src_labels)))
ax.set_yticklabels(src_labels)
ax.set_title('Missing Value Percentage (%)', pad=12)
cbar = fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
cbar.set_label('Missing %')
for i in range(len(src_labels)):
    for j in range(len(fields)):
        ax.text(j, i, f'{mv_matrix[i, j]:.2f}%', ha='center', va='center',
                color='white' if mv_matrix[i, j] > 5 else '#AAAACC', fontsize=10, fontweight='bold')
plt.tight_layout()
plt.savefig(PLOTS / 'missing_values.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: missing_values.png')

---
## 4. Business Name Analysis

In [ ]:
def name_stats(series, label):
    s = series.dropna().astype(str).str.strip()
    lens = s.str.len()
    print(f'\n── {label} ──')
    print(f'  Count:         {len(s):,}')
    print(f'  Unique names:  {s.nunique():,}')
    print(f'  Duplicate %:   {(1 - s.nunique()/len(s))*100:.2f}%')
    print(f'  Avg length:    {lens.mean():.2f}')
    print(f'  Median length: {lens.median():.2f}')
    print(f'  Min / Max:     {lens.min()} / {lens.max()}')
    return lens

l1 = name_stats(s1_tr['business_name'], 'Source 1 Train')
l2 = name_stats(s2_tr['business_name'], 'Source 2 Train')
l3 = name_stats(s3_tr['business_name'], 'Source 3 Train')
    
# Plot
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
for ax, (lens, label, color) in zip(axes, [
    (l1, 'Source 1 (Train)', PALETTE[0]),
    (l2, 'Source 2 (Train)', PALETTE[1]),
    (l3, 'Source 3 (Train)', PALETTE[2]),
]):
    ax.hist(lens.clip(upper=150), bins=50, color=color, alpha=0.85, edgecolor='none')
    ax.axvline(lens.median(), color='#FFBE0B', lw=1.5, ls='--', label=f'Median={lens.median():.0f}')
    ax.set_title(label)
    ax.set_xlabel('Name length (chars)')
    ax.set_ylabel('Count')
    ax.legend(fontsize=8, framealpha=0.3)
    ax.yaxis.set_major_formatter(mtick.FuncFormatter(lambda x, _: f'{int(x):,}'))
    ax.grid(True, axis='y')
fig.suptitle('Business Name Length Distribution', fontsize=15, y=1.01)
plt.tight_layout()
plt.savefig(PLOTS / 'name_length.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: name_length.png')

In [ ]:
# Legal suffix noise patterns
import re
all_names = pd.concat([s1_tr['business_name'], s2_tr['business_name'], s3_tr['business_name']]).dropna().astype(str)

patterns = {
    'Corp / Corporation': r'\b(corp|corporation)\b',
    'Pvt / Private':      r'\b(pvt|private)\b',
    'Ltd / Limited':      r'\b(ltd|limited)\b',
    'Inc / Incorporated': r'\b(inc|incorporated)\b',
    'LLC':                r'\bllc\b',
    'Ampersand (&)':      r'&',
}

print('Legal suffix noise counts (all train sources combined):')
for label, pat in patterns.items():
    cnt = int(all_names.str.contains(pat, case=False, regex=True).sum())
    print(f'  {label:30s}: {cnt:>10,}')

---
## 5. Business Address Analysis

In [ ]:
def addr_stats(series, label):
    s = series.dropna().astype(str).str.strip()
    lens = s.str.len()
    has_digit = s.str.contains(r'\d', regex=True)
    has_post  = s.str.contains(r'\b\d{5,6}\b', regex=True)
    print(f'\n── {label} ──')
    print(f'  Count:           {len(s):,}')
    print(f'  Avg length:      {lens.mean():.2f}')
    print(f'  Median length:   {lens.median():.2f}')
    print(f'  Min / Max:       {lens.min()} / {lens.max()}')
    print(f'  Has numeric %:   {has_digit.mean()*100:.2f}%')
    print(f'  Has postcode %:  {has_post.mean()*100:.2f}%')
    return lens

a1 = addr_stats(s1_tr['business_address'], 'Source 1 Train')
a2 = addr_stats(s2_tr['business_address'], 'Source 2 Train')
a3 = addr_stats(s3_tr['business_address'], 'Source 3 Train')

# Plot
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
for ax, (lens, label, color) in zip(axes, [
    (a1, 'Source 1 (Train)', PALETTE[0]),
    (a2, 'Source 2 (Train)', PALETTE[1]),
    (a3, 'Source 3 (Train)', PALETTE[2]),
]):
    ax.hist(lens.clip(upper=300), bins=50, color=color, alpha=0.85, edgecolor='none')
    ax.axvline(lens.median(), color='#FFBE0B', lw=1.5, ls='--', label=f'Median={lens.median():.0f}')
    ax.set_title(label)
    ax.set_xlabel('Address length (chars)')
    ax.set_ylabel('Count')
    ax.legend(fontsize=8, framealpha=0.3)
    ax.yaxis.set_major_formatter(mtick.FuncFormatter(lambda x, _: f'{int(x):,}'))
    ax.grid(True, axis='y')
fig.suptitle('Business Address Length Distribution', fontsize=15, y=1.01)
plt.tight_layout()
plt.savefig(PLOTS / 'address_length.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: address_length.png')

In [ ]:
# Address abbreviation analysis
all_addr = pd.concat([s1_tr['business_address'], s2_tr['business_address'], s3_tr['business_address']]).dropna().astype(str)

abbrev_patterns = {
    'Rd (abbrev)':    r'\brd\b',
    'Road (full)':    r'\broad\b',
    'St (abbrev)':    r'\bst\b',
    'Street (full)':  r'\bstreet\b',
}

print('Address abbreviation noise counts (all train sources combined):')
for label, pat in abbrev_patterns.items():
    cnt = int(all_addr.str.contains(pat, case=False, regex=True).sum())
    print(f'  {label:20s}: {cnt:>10,}')

---
## 6. Country Distribution & Open-Set Verification

In [ ]:
# Country counts per source
print('=== TRAIN country distribution ===')
for label, df in [('S1', s1_tr), ('S2', s2_tr), ('S3', s3_tr)]:
    vc = df['country'].fillna('MISSING').value_counts()
    print(f'  {label}: {vc.to_dict()}')

print('\n=== TEST country distribution ===')
for label, df in [('S1', s1_te), ('S2', s2_te), ('S3', s3_te)]:
    vc = df['country'].fillna('MISSING').value_counts()
    print(f'  {label}: {vc.to_dict()}')

# Unique countries
train_countries = sorted(set(s1_tr['country'].dropna()) | set(s2_tr['country'].dropna()) | set(s3_tr['country'].dropna()))
test_countries  = sorted(set(s1_te['country'].dropna()) | set(s2_te['country'].dropna()) | set(s3_te['country'].dropna()))
print(f'\nUnique countries in TRAIN: {train_countries}')
print(f'Unique countries in TEST:  {test_countries}')
print(f'France in TEST: {"France" in test_countries}')

In [ ]:
# Plot: Country Distribution
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

for ax, (title, dfs) in zip(axes, [
    ('Train (S1+S2+S3)', [s1_tr, s2_tr, s3_tr]),
    ('Test  (S1+S2+S3)', [s1_te, s2_te, s3_te]),
]):
    combined = pd.concat([d['country'] for d in dfs]).fillna('MISSING')
    vc = combined.value_counts()
    bars = ax.barh(vc.index.tolist(), vc.values, color=PALETTE[:len(vc)], alpha=0.9)
    ax.set_title(title)
    ax.set_xlabel('Number of records')
    ax.xaxis.set_major_formatter(mtick.FuncFormatter(lambda x, _: f'{int(x):,}'))
    for bar, val in zip(bars, vc.values):
        ax.text(val * 1.01, bar.get_y() + bar.get_height() / 2,
                f'{val:,}', va='center', color='#E0E0FF', fontsize=9)
    ax.grid(True, axis='x')

fig.suptitle('Country Distribution — Train vs Test', fontsize=15, y=1.01)
plt.tight_layout()
plt.savefig(PLOTS / 'country_distribution.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: country_distribution.png')

---
## 7. Ground Truth — Match Distribution

In [ ]:
def parse_matched(val):
    if pd.isna(val) or str(val).strip() == '': return []
    return [x.strip() for x in str(val).split(',') if x.strip()]

gt_tr['match_list']  = gt_tr['matched_entity_ids'].apply(parse_matched)
gt_tr['match_count'] = gt_tr['match_list'].apply(len)

print('Ground truth match count distribution:')
print(f'  Zero matches (singletons): {(gt_tr["match_count"]==0).sum():,}')
print(f'  One match:                 {(gt_tr["match_count"]==1).sum():,}')
print(f'  Multiple matches (>=2):    {(gt_tr["match_count"]>=2).sum():,}')
print(f'  Average matches per S1:    {gt_tr["match_count"].mean():.3f}')
print(f'  Maximum matches for one S1:{gt_tr["match_count"].max()}')

In [ ]:
# Source category analysis: S2-only / S3-only / both
def src_category(lst):
    has_s2 = any(x.startswith('S2-') for x in lst)
    has_s3 = any(x.startswith('S3-') for x in lst)
    if has_s2 and has_s3: return 'both S2 & S3'
    if has_s2:            return 'S2 only'
    if has_s3:            return 'S3 only'
    return 'none'

matched_rows = gt_tr[gt_tr['match_count'] > 0].copy()
matched_rows['src_cat'] = matched_rows['match_list'].apply(src_category)
print('Source category breakdown (among entities with ≥1 match):')
print(matched_rows['src_cat'].value_counts().to_string())

---
## 8. Load Pre-Computed Stats & Display Final Summary

In [ ]:
stats_path = OUT / 'eda_stats.json'
if stats_path.exists():
    with open(stats_path) as f:
        stats = json.load(f)
    print('Pre-computed stats loaded from eda_stats.json')
    print(json.dumps(stats['noise_patterns'], indent=2))
else:
    print('Run eda_script.py first to generate eda_stats.json')

---
## 9. Key Conclusions for Normalization & Blocking Design

| Finding | Implication |
|---|---|
| Legal suffix abbreviations confirmed (Corp/Ltd/Pvt) | Canonicalize suffixes before string comparison |
| Address abbreviations coexist with full forms (Rd/Road, St/Street) | Use token Jaccard / TF-IDF for address matching, not exact match |
| Postal code presence is inconsistent across sources | Cannot be a mandatory blocking key; use as confirming feature |
| France appears in test but not training | Country must be an open-set string label — never one-hot encode |
| Significant singleton fraction in ground truth | Model must confidently predict empty matches; false merges are 2× penalized |
| Many S1 entities match both S2 and S3 records | Blocking must cover both sources simultaneously |

**Recommended blocking strategy (informed by this EDA):**
1. Token-level inverted index on business name (highest recall ceiling)
2. Phonetic (Soundex/Metaphone) key blocking to capture typos and transliterations
3. Postal code co-blocking where available (high precision when present)
4. Country pre-filter as a pre-blocking gate (string equality only, open-set safe)